# 2D Toric Code — Exact Diagonalization + NQS (live ΔE) on Colab

Two-stage notebook for the **perturbed 2D toric / surface code**
`H = -J Σ_v A_v - J Σ_p B_p - h_x Σ σˣ - h_z Σ σᶻ`, reusing the repo's own
modules (`model/`, `simulation/`) — same geometry, Hamiltonian, CNN ansatz and
vertex-flip sampler as `main.py`.

**How to use**
1. Run **cells 1–3** once (clone repo, install pinned deps, imports).
2. **Step 1 — ED**: set the physics (`L, BC, hx, hy, hz, J`) and get the exact
   ground-state energy `E_exact` (+ gap). Small systems only (dim = 2ᴺ).
3. **Step 2 — NQS**: set every hyperparameter (`dt, diag_shift, n_samples,
   n_sweeps, n_chains, …`) and train. Each step prints **live** `E`, the figure
   of merit **ΔE = |E − E_exact|/|E_exact|**, plus `R̂`, `τ_corr`, `V-score`,
   and `mcmc_acc`. Re-run Step 2 alone to try new hyperparameters; it reuses
   `E_exact` from Step 1 (re-run Step 1 if you change the physics).

> GPU is **not** needed at these sizes (L=3 OBC = 12 qubits); CPU runtime is fine.
> N (qubits): OBC = 2·L² − 2L, PBC = 2·L².  Keep N ≲ 22 for the ED cell.


In [ ]:
# 1. Clone (or update) the repo into the Colab VM. Idempotent: safe to re-run.
import os
REPO_URL = "https://github.com/SanzharBissenali/ThreeD_TC.git"
REPO_DIR = "/content/ThreeD_TC"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git pull --ff-only || true
print("cwd:", os.getcwd())

In [ ]:
# 2. Install the NQS stack, pinned to the repo's requirements.txt (result parity).
#    numba is needed by model/exact_diag.py. Colab may print harmless conflict
#    warnings for unrelated preinstalled packages. CPU is enough for L=3.
!pip install -q jax==0.5.2 jaxlib==0.5.1 netket==3.16.1.post1 flax==0.10.4 \
    numpy==2.1.3 scipy==1.15.2 numba tqdm matplotlib

In [ ]:
# 3. Imports + float64. NetKet needs x64; set it before importing jax/netket.
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import netket as nk
from functools import partial
from scipy.sparse.linalg import eigsh
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Repo modules (the SAME code main.py uses)
from model.geometry import ToricCodeGeometry
from model.hamiltonian import create_hamiltonian
from model.exact_diag import hamiltonian_linop
from model.networks import KernelManager, create_model
from simulation.custom_sampler import create_custom_sampler

print("jax", jax.__version__, "| netket", nk.__version__, "| backend:", jax.default_backend())

## Step 1 — Exact diagonalization

Builds the matrix-free Hamiltonian (`model/exact_diag.py`, Numba) for the chosen
physical point and returns the exact ground-state energy `E_exact` (and the gap
Δ = E₁ − E₀). **Edit the physics here.** These same values feed Step 2.

In [ ]:
# === EDIT: physical point (shared by ED and NQS) ==========================
L   = 3          # vertices per side
BC  = "OBC"      # "OBC" | "PBC"
hx  = 0.2        # sigma^x field
hy  = 0.0        # sigma^y field (nonzero -> sign problem -> complex ansatz)
hz  = 0.5        # sigma^z field
J   = 1.0        # toric-code coupling
# ==========================================================================

geo   = ToricCodeGeometry(L, L, BC)
N     = geo.N
DTYPE = "complex" if hy != 0.0 else "float64"
PHYS  = dict(L=L, BC=BC, hx=hx, hy=hy, hz=hz, J=J)   # snapshot for Step 2
assert N <= 24, f"N={N} -> dim 2^{N} too large for ED in Colab. Use smaller L/PBC->OBC."

H_ed, _ = hamiltonian_linop(geo, hx=hx, hy=hy, hz=hz, J=J)
k = 2 if (1 << N) > 2 else 1
evals = np.sort(eigsh(H_ed, k=k, which="SA", return_eigenvectors=False))
E_exact = float(evals[0])
gap     = float(evals[1] - evals[0]) if k > 1 else float("nan")

print(f"system: L={L} {BC}  ->  N = {N} qubits   (Hilbert dim = 2^{N} = {1<<N})")
print(f"point : hx={hx}  hy={hy}  hz={hz}  J={J}")
print(f"E_exact       = {E_exact:.8f}")
print(f"E_exact / site = {E_exact / N:.8f}")
print(f"gap (E1 - E0) = {gap:.6f}")

## Step 2 — NQS (variational Monte Carlo + Stochastic Reconfiguration)

Same ansatz/sampler as `main.py`. Set **every** hyperparameter below. The loop is
the repo's SR step (dense QGT + `pinv_smooth`, `model/`/`simulation/`), printing
live every iteration:

- `E` ± error, and **`ΔE = |E − E_exact|/|E_exact|`** (the figure of merit)
- `R̂` (Gelman–Rubin; want ≈ 1), `τ_corr` (autocorr time), `V = N·Var/E²` (V-score)
- `mcmc_acc` (Metropolis acceptance — phase-dependent, not a health metric)

Re-run **this cell only** to try new hyperparameters (reuses `E_exact`).

In [ ]:
# === EDIT: NQS hyperparameters ===========================================
# -- architecture (Combo: non-invariant CNN -> Wilson -> invariant CNN) --
ARCH            = "Combo"        # "Combo" | "RPP"
CHANNELS_NONINV = [1, 2, 4]      # non-invariant block channels (in,...,out)
CHANNELS_INV    = [4, 4, 4]      # invariant block channels (first == NONINV out)
KERNEL_SIZE     = 2              # non-inv kernel size (saturates at L-1; L=3 -> 2==3)
RESCALE         = 10 ** 1.5      # Wilson-nonlinearity rescale

# -- optimization (SR) --
N_ITER          = 150            # number of SR steps
DT              = 7e-3           # step size == effective learning rate
DIAG_SHIFT      = 1e-3           # QGT regularization (raise if unstable)

# -- Monte Carlo sampling --
N_SAMPLES       = 1024           # total MC samples per step
N_CHAINS        = 16             # Metropolis chains (keep >= a few hundred samp/chain)
N_SWEEPS        = N // 2         # sweeps between recorded samples (main.py default N/2)
N_DISCARD       = 8              # warmup samples discarded per chain
CHUNK_SIZE      = 1024           # gradient chunking (memory)
USE_CUSTOM_SAMPLER = True        # True -> 75% local + 25% vertex(star)-flip rule
SEED            = 0
# ==========================================================================

assert "E_exact" in globals(), "Run the ED cell (Step 1) first."
np.random.seed(SEED)

config = dict(architecture=ARCH, channels_noninv=CHANNELS_NONINV,
              channels_inv=CHANNELS_INV, rescale=RESCALE, dtype=DTYPE, bc=BC,
              Lx=L, Ly=L, kernel_size=KERNEL_SIZE, kernel_size_inv=L - 1,
              n_chains=N_CHAINS, n_sweeps=N_SWEEPS)

hi = nk.hilbert.Spin(s=1/2, N=N)
H  = create_hamiltonian(hi, geo.vertex_all, geo.plaq_all, geo.bonds,
                        hx=hx, hy=hy, hz=hz, J=J, dtype=DTYPE)
km    = KernelManager(L, L, BC, KERNEL_SIZE, L - 1, geo.arr_coord, geo.dg_p, N)
model = create_model(config, geo.plaq_all, km)

if USE_CUSTOM_SAMPLER:
    sa = create_custom_sampler(geo, hi, config)
else:
    sa = nk.sampler.MetropolisSampler(hi, rule=nk.sampler.rules.LocalRule(),
                                      n_chains=N_CHAINS, n_sweeps=N_SWEEPS, dtype=jnp.int8)
vs = nk.vqs.MCState(sa, model, n_samples=N_SAMPLES,
                    n_discard_per_chain=N_DISCARD, chunk_size=CHUNK_SIZE)

print(f"n_params = {vs.n_parameters} | N = {N} | sampler = "
      f"{'vertex-flip (75/25)' if USE_CUSTOM_SAMPLER else 'local'} | E_exact = {E_exact:.6f}")

hist = {k: [] for k in ["iter", "E", "E_err", "deltaE", "R_hat", "tau_corr", "Vscore", "mcmc_acc"]}
loop = tqdm(range(N_ITER))
for step in loop:
    E, f = vs.expect_and_grad(H)
    S = vs.quantum_geometric_tensor(
        nk.optimizer.qgt.QGTJacobianDense(diag_shift=DIAG_SHIFT, diag_scale=0.0))
    gamma_f = jax.tree.map(lambda x: -1.0 * x, f)
    dtheta, _ = S.solve(partial(nk.optimizer.solver.pinv_smooth,
                                rtol=1e-30, rtol_smooth=1e-30), gamma_f)
    vs.parameters = jax.tree.map(lambda a, b: a + DT * b, vs.parameters, dtheta)

    Em   = float(np.real(E.mean))
    dE   = abs(Em - E_exact) / abs(E_exact)
    acc  = float(vs.sampler_state.n_accepted) / float(vs.sampler_state.n_steps)
    vsc  = N * float(E.variance) / Em**2
    for key, val in zip(hist, [step, Em, float(E.error_of_mean), dE,
                               float(E.R_hat), float(E.tau_corr), vsc, acc]):
        hist[key].append(val)
    if np.isnan(Em):
        print("NaN energy — stopping."); break
    loop.set_description(
        f"E={Em:.5f}±{float(E.error_of_mean):.4f} | dE={dE:.2e} | "
        f"R_hat={float(E.R_hat):.3f} | tau={float(E.tau_corr):.2f} | "
        f"V={vsc:.2e} | acc={acc:.3f}")

# --- final summary (median of last 15 steps -> robust to MC noise) ---
tail = slice(-15, None)
E_fin  = float(np.median(hist["E"][tail]))
dE_fin = abs(E_fin - E_exact) / abs(E_exact)
print("\n=== FINAL (median of last 15 steps) ===")
print(f"E_NQS   = {E_fin:.6f}      E_exact = {E_exact:.6f}")
print(f"deltaE  = {dE_fin:.3e}")
print(f"R_hat   = {np.median(hist['R_hat'][tail]):.3f}   "
      f"tau_corr = {np.median(hist['tau_corr'][tail]):.2f}   "
      f"V-score = {np.median(hist['Vscore'][tail]):.2e}   "
      f"mcmc_acc = {np.median(hist['mcmc_acc'][tail]):.3f}")

In [ ]:
# Convergence plots: energy vs E_exact, and deltaE (log scale).
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist["iter"], hist["E"], label="E_NQS")
ax[0].axhline(E_exact, color="k", ls="--", label="E_exact")
ax[0].set_xlabel("SR step"); ax[0].set_ylabel("energy"); ax[0].legend(); ax[0].set_title("energy")
ax[1].semilogy(hist["iter"], hist["deltaE"])
ax[1].set_xlabel("SR step"); ax[1].set_ylabel(r"$\Delta E=|E-E_{exact}|/|E_{exact}|$")
ax[1].set_title("relative error (log)"); ax[1].grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

## Notes

- **Sweeping hyperparameters:** edit the Step-2 cell and re-run it — `E_exact`
  persists from Step 1. Change the physical point only in Step 1 (then re-run both).
- **Non-invariant `KERNEL_SIZE`** saturates the receptive field at `L-1`: at L=3,
  size 2 already covers the whole system, so size 3 is the identical network.
- **`mcmc_acc` is phase-dependent**, not a health metric: it collapses toward the
  hz-dominated trivial phase (z-polarized state → peaked in the z-sampling basis).
  Judge convergence by `R̂ ≈ 1`, stable `τ_corr`, and `ΔE`/`V-score`.
- **ED size limit:** dim = 2ᴺ. L=3 OBC = 12 qubits (4096) is instant; L=4 OBC = 24
  qubits (16.7 M) needs a high-RAM runtime and is the practical ceiling here.
